# Runnable Configuration Utilities

This document covers the developer-facing top-level statements defined in `langchain_core.runnables.config`.

# `RunnableConfig: TypedDict(total=False)`

`RunnableConfig` stores runtime settings supplied when invoking, batching, streaming, or composing a `Runnable`.

All fields are optional because configurations may be created partially and merged with inherited or call-time configuration.

## Fields

```python
tags: list[str] # Tags inherited by this call and its child calls
metadata: dict[str, Any] # JSON-serializable metadata inherited by child calls
callbacks: Callbacks # Callback handlers or callback manager used during execution
run_name: str # Custom tracer name for the current Runnable call
max_concurrency: int | None # Maximum number of parallel operations
recursion_limit: int # Maximum nested Runnable execution depth
configurable: dict[str, Any] # Runtime values for configurable fields or alternatives
run_id: uuid.UUID | None # Optional unique identifier for the tracer run
```

# `ContextThreadPoolExecutor: ThreadPoolExecutor`

`ContextThreadPoolExecutor` is a thread-pool executor that copies the current Python context into worker threads.

This preserves values stored in `ContextVar` objects, including inherited Runnable configuration.

It uses the constructor inherited from `ThreadPoolExecutor`.

## Overridden Methods

### `submit`

Submits one callable while running it inside a copy of the current context.

### `map`

Applies a callable to multiple iterables while providing copied context to worker executions.

# `set_config_context`

Temporarily places a `RunnableConfig` into the child Runnable context and synchronizes the tracing context.

**Syntax**

```python
set_config_context(
    config: RunnableConfig, # Configuration stored in the current context
) -> Generator[Context, None, None] # Yield the copied execution context
```

This function is used as a context manager.

# `ensure_config`

Creates a complete normalized `RunnableConfig`.

It merges inherited child configuration, recognized call-time fields, custom configurable values, and default values.

**Syntax**

```python
ensure_config(
    config: RunnableConfig | None = None, # Optional partial configuration
) -> RunnableConfig # Return the normalized configuration
```

# `get_config_list`

Creates one normalized configuration for every input in a batch.

A single configuration is duplicated, while a sequence of configurations must match the requested length.

**Syntax**

```python
get_config_list(
    config: RunnableConfig | Sequence[RunnableConfig] | None, # One configuration or one configuration per input
    length: int, # Number of configurations required
) -> list[RunnableConfig] # Return normalized configurations for the batch
```

It raises `ValueError` when `length` is negative or when the number of supplied configurations does not match `length`.

When a single configuration contains `run_id`, that identifier is retained only for the first batch item.

# `patch_config`

Returns a normalized configuration with selected values replaced or extended.

**Syntax**

```python
patch_config(
    config: RunnableConfig | None, # Original configuration
    *,
    callbacks: BaseCallbackManager | None = None, # Replacement callback manager
    recursion_limit: int | None = None, # Replacement recursion limit
    max_concurrency: int | None = None, # Replacement concurrency limit
    run_name: str | None = None, # Replacement run name
    configurable: dict[str, Any] | None = None, # Configurable values merged into existing values
) -> RunnableConfig # Return the patched configuration
```

Replacing callbacks removes an existing `run_name` and `run_id` because those values belong to the original run.

# `merge_configs`

Combines multiple configurations into one configuration.

**Syntax**

```python
merge_configs(
    *configs: RunnableConfig | None, # Configurations merged from left to right
) -> RunnableConfig # Return the merged configuration
```

Tags are combined without duplicates, metadata and configurable values use later values on conflicts, and callback collections or managers are combined.

A non-default recursion limit replaces the default recursion limit.

# `call_func_with_variable_args`

Synchronously calls a function that may optionally accept `config`, `run_manager`, or both.

**Syntax**

```python
call_func_with_variable_args(
    func: Callable[[Input], Output]
    | Callable[[Input, RunnableConfig], Output]
    | Callable[[Input, CallbackManagerForChainRun], Output]
    | Callable[[Input, CallbackManagerForChainRun, RunnableConfig], Output], # Function receiving input and optional LangChain arguments
    input: Input, # Input passed to the function
    config: RunnableConfig, # Runtime configuration passed when supported
    run_manager: CallbackManagerForChainRun | None = None, # Optional callback run manager
    **kwargs: Any, # Additional keyword arguments passed to the function
) -> Output # Return the function output
```

A child callback manager is inserted into the configuration when both configuration and a run manager are supported.

# `acall_func_with_variable_args`

Calls an asynchronous function that may optionally accept `config`, `run_manager`, or both.

**Syntax**

```python
acall_func_with_variable_args(
    func: Callable[[Input], Awaitable[Output]]
    | Callable[[Input, RunnableConfig], Awaitable[Output]]
    | Callable[[Input, AsyncCallbackManagerForChainRun], Awaitable[Output]]
    | Callable[[Input, AsyncCallbackManagerForChainRun, RunnableConfig], Awaitable[Output]], # Async function receiving input and optional LangChain arguments
    input: Input, # Input passed to the function
    config: RunnableConfig, # Runtime configuration passed when supported
    run_manager: AsyncCallbackManagerForChainRun | None = None, # Optional asynchronous callback run manager
    **kwargs: Any, # Additional keyword arguments passed to the function
) -> Awaitable[Output] # Return the function awaitable
```

# `get_callback_manager_for_config`

Creates a synchronous callback manager from a Runnable configuration.

**Syntax**

```python
get_callback_manager_for_config(
    config: RunnableConfig, # Configuration containing callbacks, tags, and metadata
) -> CallbackManager # Return the configured synchronous callback manager
```

# `get_async_callback_manager_for_config`

Creates an asynchronous callback manager from a Runnable configuration.

**Syntax**

```python
get_async_callback_manager_for_config(
    config: RunnableConfig, # Configuration containing callbacks, tags, and metadata
) -> AsyncCallbackManager # Return the configured asynchronous callback manager
```

# `get_executor_for_config`

Creates a context-aware executor using the concurrency limit stored in a Runnable configuration.

**Syntax**

```python
get_executor_for_config(
    config: RunnableConfig | None, # Configuration containing an optional maximum concurrency
) -> Generator[Executor, None, None] # Yield the configured executor
```

This function is used as a context manager and closes the executor when the context exits.

# `run_in_executor`

Asynchronously runs a synchronous callable in an executor.

**Syntax**

```python
async run_in_executor(
    executor_or_config: Executor | RunnableConfig | None, # Explicit executor, Runnable configuration, or default executor selection
    func: Callable[P, T], # Synchronous callable executed by the worker
    *args: P.args, # Positional arguments passed to the callable
    **kwargs: P.kwargs, # Keyword arguments passed to the callable
) -> T # Return the callable result
```

When a configuration or `None` is supplied, the default event-loop executor is used with the current context copied into the worker.

A `StopIteration` raised by the callable is converted into `RuntimeError` so that the asynchronous future does not remain pending.

In [1]:
from langchain_core.runnables import RunnableLambda # Import RunnableLambda
from langchain_core.runnables.config import RunnableConfig # Import the configuration type
from langchain_core.runnables.config import ensure_config # Import configuration normalizer
from langchain_core.runnables.config import merge_configs # Import configuration merger
from langchain_core.runnables.config import patch_config # Import configuration updater

def create_message(name: str, config: RunnableConfig) -> str: # Define a function that receives input and configuration
    prefix = config["configurable"].get("prefix", "Hello") # Read a configurable runtime value
    request_id = config["metadata"].get("request_id", "unknown") # Read metadata from the configuration
    return f"{prefix}, {name}! Request ID: {request_id}" # Return the generated message

runnable = RunnableLambda(create_message) # Convert the function into a Runnable

base_config = ensure_config( # Create a complete normalized configuration
    {
        "tags": ["greeting"], # Add a tracing tag
        "metadata": {"request_id": 101}, # Add metadata
        "configurable": {"prefix": "Welcome"}, # Add a configurable runtime value
    }
)

additional_config = ensure_config( # Create another normalized configuration
    {
        "tags": ["jupyter"], # Add another tracing tag
        "metadata": {"source": "notebook"}, # Add additional metadata
    }
)

merged_config = merge_configs(base_config, additional_config) # Merge both configurations

final_config = patch_config( # Update selected configuration values
    merged_config, # Provide the configuration to update
    run_name="greeting_run", # Set the Runnable execution name
    max_concurrency=2, # Set the maximum parallel-execution limit
)

result = runnable.invoke("Saad", config=final_config) # Execute the Runnable with the final configuration

print(result) # Display the returned message

Welcome, Saad! Request ID: 101
